In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
rfm = pd.read_csv('../data/processed/rfm_log_kmeans.csv')
rfm.head()

,Recency,Frequency,Monetary_log,AvgOrderValue_log,KMeans_Cluster
0,165,2,5.102363,4.415280,2
1,3,2,7.188654,6.496262,3
2,74,1,5.403398,5.403398,1
3,43,2,7.706226,7.013529,3
4,11,1,5.710195,5.710195,3


### Split data to detect anomly at VIP

In [3]:
VIP_CLUSTER_ID = 3

features = [
    "Recency",
    "Frequency",
    "Monetary_log",
    "AvgOrderValue_log"
]

vip_df = rfm[rfm["KMeans_Cluster"] == VIP_CLUSTER_ID][features].copy()

print("Number of VIP customers:", vip_df.shape[0])
vip_df.head()

Number of VIP customers: 2131


,Recency,Frequency,Monetary_log,AvgOrderValue_log
1,3,2,7.188654,6.496262
3,43,2,7.706226,7.013529
4,11,1,5.710195,5.710195
5,11,2,5.842965,5.152713
6,44,1,5.764438,5.764438


### Train Isolation forest model

In [6]:
from sklearn.ensemble import IsolationForest

iso_vip = IsolationForest(
    n_estimators=200,
    contamination=0.02,  
    random_state=42
)
iso_vip.fit(vip_df)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.02
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [7]:
vip_df["anomaly_flag"] = iso_vip.predict(vip_df)

vip_df["anomaly_flag"].value_counts(normalize=True) * 100

anomaly_flag
 1    97.982168
-1     2.017832
Name: proportion, dtype: float64

In [8]:
anomalies = vip_df[vip_df["anomaly_flag"] == -1]
normals = vip_df[vip_df["anomaly_flag"] == 1]

anomalies.describe()

,Recency,Frequency,Monetary_log,AvgOrderValue_log,anomaly_flag
count,43.000000,43.000000,43.000000,43.000000,43.0
mean,13.093023,47.348837,8.960166,6.101814,-1.0
std,14.651205,47.613965,3.082199,1.892365,0.0
min,1.000000,1.000000,2.480731,2.480731,-1.0
25%,3.000000,3.000000,8.380684,4.525838,-1.0
50%,8.000000,38.000000,10.264779,6.575567,-1.0
75%,15.500000,79.500000,10.894552,7.604485,-1.0
max,51.000000,183.000000,12.747715,9.326432,-1.0


In [9]:
import joblib
joblib.dump(iso_vip, "../artifacts/iso_vip.pkl")

['../artifacts/iso_vip.pkl']

In [10]:
OCC_CLUSTER_ID = 2

features = [
    "Recency",
    "Frequency",
    "Monetary_log",
    "AvgOrderValue_log"
]

occ_df = rfm[rfm["KMeans_Cluster"] == OCC_CLUSTER_ID][features].copy()

print("Occasional customers count:", occ_df.shape[0])
occ_df.head()

Occasional customers count: 611


,Recency,Frequency,Monetary_log,AvgOrderValue_log
0,165,2,5.102363,4.415280
7,203,1,6.192792,6.192792
23,198,1,7.250423,7.250423
32,227,1,5.462772,5.462772
39,157,1,6.321955,6.321955


In [11]:
iso_occ = IsolationForest(
    n_estimators=200,
    contamination=0.05,  
    random_state=42
)

iso_occ.fit(occ_df)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [12]:
occ_df["anomaly_flag"] = iso_occ.predict(occ_df)

occ_df["anomaly_flag"].value_counts(normalize=True) * 100

anomaly_flag
 1    94.92635
-1     5.07365
Name: proportion, dtype: float64

In [13]:
anomalies = occ_df[occ_df["anomaly_flag"] == -1]
normals = occ_df[occ_df["anomaly_flag"] == 1]

anomalies.describe()

,Recency,Frequency,Monetary_log,AvgOrderValue_log,anomaly_flag
count,31.000000,31.000000,31.000000,31.000000,31.0
mean,193.225806,5.806452,6.000505,4.992766,-1.0
std,38.599404,8.791736,2.996686,2.314836,0.0
min,138.000000,1.000000,0.936093,0.936093,-1.0
25%,158.000000,1.000000,2.999270,2.972159,-1.0
50%,186.000000,2.000000,7.578652,4.831735,-1.0
75%,233.500000,7.000000,8.586128,6.969077,-1.0
max,247.000000,41.000000,9.996191,9.294514,-1.0


In [14]:
joblib.dump(iso_occ, "../artifacts/iso_occ.pkl")

['../artifacts/iso_occ.pkl']

In [15]:
REG_CLUSTER_ID = 1

features = [
    "Recency",
    "Frequency",
    "Monetary_log",
    "AvgOrderValue_log"
]

reg_df = rfm[rfm["KMeans_Cluster"] == REG_CLUSTER_ID][features].copy()

print("Number of Regular customers:", reg_df.shape[0])
reg_df.head()

Number of Regular customers: 1102


,Recency,Frequency,Monetary_log,AvgOrderValue_log
2,74,1,5.403398,5.403398
11,61,6,7.849464,6.059653
13,98,3,5.625280,4.533853
20,57,2,7.624272,6.931613
24,58,2,6.542443,5.850736


In [16]:
iso_regular = IsolationForest(
    n_estimators=200,
    contamination=0.03,   # ~3% anomalies
    random_state=42
)

iso_regular.fit(reg_df)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.03
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [17]:
reg_df["anomaly_flag"] = iso_regular.predict(reg_df)

reg_df["anomaly_flag"].value_counts(normalize=True) * 100

anomaly_flag
 1    96.914701
-1     3.085299
Name: proportion, dtype: float64

In [18]:
anomalies = reg_df[reg_df["anomaly_flag"] == -1]
normals = reg_df[reg_df["anomaly_flag"] == 1]

anomalies.describe

<bound method NDFrame.describe of       Recency  Frequency  Monetary_log  AvgOrderValue_log  anomaly_flag
42         56          4      9.421219           8.035167            -1
303        63         41      8.704902           4.997939            -1
416        60         18      8.799155           5.911345            -1
452        65          1      3.433987           3.433987            -1
460       128         10      8.365074           6.064583            -1
539        67          1      3.234749           3.234749            -1
677        53          5      9.338301           7.729215            -1
909        56          2      3.770459           3.100092            -1
918       107          8      8.950053           6.871519            -1
922        74          1      9.382766           9.382766            -1
1210       98          1      3.712352           3.712352            -1
1225       71         21      8.889067           5.847299            -1
1247      122          1      

In [19]:
joblib.dump(iso_regular , "../artifacts/iso_reg.pkl")

['../artifacts/iso_reg.pkl']